# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sumit07-git/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
con = duckdb.connect()

In [20]:
con.execute(f"""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB Hugging Face authentication configured")

DuckDB Hugging Face authentication configured


Unit of analysis: One row represents the daily performance of one content item for one pseudonymized client on one report date.

Time window: I will use March 2026 (`month = '2026-03'`) as the development window. I will not use the June 2026 `_sample` table for label development because it is the final month and should remain a sealed outcome window.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

con.sql(query).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


### Feature

I will use historical performance fields that would be available at the decision moment. These include previous-period impressions, clicks, sessions, and search-position information. I will use no more than five features.

### Label / proxy

My label will represent a future performance outcome after the decision date. It will be constructed only from the future outcome window and will not be used as an input feature.

### Context

`report_date`, `client_hash_id`, and `content_hash_id` are context fields. They identify the observation, client, and content item and can be used for grouping, joining, and time-aware validation, but they are not model features.

### Excluded

I will exclude future performance fields because they would not be known at the decision moment. I will also exclude label-derived fields such as `trend_pct` or `trend_direction` when they encode the outcome being predicted, because using them would create target leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
schema_query = """
DESCRIBE
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
"""

schema_df = con.sql(schema_query).df()

schema_df

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


I use March 2026 as the development month. I verify three things: the stated row grain, the number of observed rows and date span, and the availability of rows using the warehouse availability flag. The queries below use only the March development slice and keep June 2026 as the sealed final month.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
query1 = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.sql(query1).df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [24]:
query2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""

count_date_check = con.sql(query2).df()

count_date_check

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [25]:
query3 = """
SELECT
    COUNT(*) AS available_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE ga4_data_available IS TRUE
"""

availability_check = con.sql(query3).df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


### Data limitation

**Uneven historical coverage:** Different clients have different amounts of historical data. Therefore, the same calendar window does not necessarily provide the same amount of usable history for every client. This can affect the availability and comparability of historical features across clients.

**GA4 availability:** GA4-dependent fields are not available for every row. Before a client's GA4 start date, some GA4 metrics may be zero-filled, so a zero value cannot always be interpreted as genuine zero activity. I therefore use the `ga4_data_available` flag when working with GA4-dependent features.

**Window overlap:** Historical feature windows and future outcome windows can overlap if the decision date and look-forward period are not defined carefully. I therefore define the decision moment first and keep future outcome information out of the feature set.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

query4 = """
SELECT
    ga4_data_available,
    COUNT(*) AS row_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
WHERE month = '2026-03'
GROUP BY ga4_data_available
ORDER BY ga4_data_available
"""

ga4_availability = con.sql(query4).df()

ga4_availability


,ga4_data_available,row_count
0,False,6408671
1,True,413966
2,<NA>,3018741


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.